<a href="https://colab.research.google.com/github/1900690/YOLO-dataset-sprit/blob/main/AI_dataset_sprit_advance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title 1. データのアップロード / ダウンロードと解凍
#@markdown データの取得方法を選択してください。（URLの場合は下の入力欄も埋めてください）

取得方法 = "URLからダウンロード" #@param ["ローカルからアップロード", "URLからダウンロード"]
zip_url = "https://github.com/1900690/YOLO-dataset-sprit/releases/download/eggplant-flower-fruit/eggplant-bbox_20260702171507.zip" #@param {type:"string"}

import os
import shutil
import urllib.request
from google.colab import files

# すでにフォルダがある場合は初期化
target_dir = "/content/datasets"
if os.path.exists(target_dir):
    shutil.rmtree(target_dir)

if 取得方法 == "URLからダウンロード":
    print("【URLからダウンロード】が選択されました。")
    if zip_url.strip() == "":
        print("エラー: URLが入力されていません。右側のフォームの zip_url にURLを入力してください。")
    else:
        file_name = zip_url.split('/')[-1]
        if not file_name.endswith('.zip'):
            file_name = "downloaded_data.zip"

        file_path = f'/content/{file_name}'
        print(f"URL: {zip_url}\nからダウンロードしています...")

        try:
            urllib.request.urlretrieve(zip_url, file_path)
            print("ダウンロード完了。解凍しています...")
            shutil.unpack_archive(file_path, '/content/')
            os.remove(file_path)
            print("データの解凍が完了しました！")
        except Exception as e:
            print(f"エラーが発生しました: {e}")

elif 取得方法 == "ローカルからアップロード":
    print("【ローカルからアップロード】が選択されました。")
    print("ZIPファイルをアップロードしてください:")
    uploaded = files.upload()

    if uploaded:
        file_name = list(uploaded.keys())[0]
        file_path = f'/content/{file_name}'
        print(f"{file_name} を解凍しています...")

        shutil.unpack_archive(file_path, '/content/')
        os.remove(file_path)
        print("データの解凍が完了しました！")
    else:
        print("ファイルがアップロードされませんでした。")

【URLからダウンロード】が選択されました。
URL: https://github.com/1900690/YOLO-dataset-sprit/releases/download/eggplant-flower-fruit/eggplant-bbox_20260702171507.zip
からダウンロードしています...
ダウンロード完了。解凍しています...
データの解凍が完了しました！


In [2]:
#@title 1.5. 画像とアノテーションの回転修正（オプション）
#@markdown 解凍されたデータの中に、向きが回転してしまっている画像がある場合、ここで修正できます。
#@markdown <br>※右に90度回転している画像を正常に戻すには、**「270」**（時計回りに270度＝反時計回りに90度）を選択してください。

修正する角度 = 270 #@param [0, 90, 180, 270] {type:"raw"}
対象の条件 = "指定した縦横ピクセルの場合のみ" #@param ["すべてのファイル", "ファイル名に特定の文字を含む場合のみ", "指定した縦横ピクセルの場合のみ"]
特定の文字 = "_rotated" #@param {type:"string"}
対象の横ピクセル_幅 = 3264 #@param {type:"integer"}
対象の縦ピクセル_高さ = 2448 #@param {type:"integer"}

import cv2
import numpy as np
import os
import glob

def rotate_image_and_bbox(image, bboxes, angle):
    """画像とYOLOフォーマット(正規化座標)のバウンディングボックスを回転する関数"""
    h, w = image.shape[:2]
    rotated_bboxes = []

    if angle == 90:
        rotated_image = cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)
        for bbox in bboxes:
            cls, x, y, bw, bh = bbox
            rotated_bboxes.append([cls, 1.0 - y, x, bh, bw])
    elif angle == 180:
        rotated_image = cv2.rotate(image, cv2.ROTATE_180)
        for bbox in bboxes:
            cls, x, y, bw, bh = bbox
            rotated_bboxes.append([cls, 1.0 - x, 1.0 - y, bw, bh])
    elif angle == 270:
        rotated_image = cv2.rotate(image, cv2.ROTATE_90_COUNTERCLOCKWISE)
        for bbox in bboxes:
            cls, x, y, bw, bh = bbox
            rotated_bboxes.append([cls, y, 1.0 - x, bh, bw])
    else:
        rotated_image = image
        rotated_bboxes = bboxes

    return rotated_image, rotated_bboxes

originals = '/content/original'
annotations = '/content/yolo/annotations'

if 修正する角度 != 0 and os.path.exists(originals):
    print(f"回転修正処理を開始します (設定角度: {修正する角度}度)...")
    img_files = glob.glob(os.path.join(originals, "*"))
    processed_count = 0

    for img_path in img_files:
        basename = os.path.basename(img_path)
        filename, ext = os.path.splitext(basename)

        if 対象の条件 == "ファイル名に特定の文字を含む場合のみ" and 特定の文字 not in filename:
            continue

        image = cv2.imread(img_path)
        if image is None:
            continue

        if 対象の条件 == "指定した縦横ピクセルの場合のみ":
            h, w = image.shape[:2]
            if w != 対象の横ピクセル_幅 or h != 対象の縦ピクセル_高さ:
                continue

        lbl_path = os.path.join(annotations, filename + ".txt")

        bboxes = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        bboxes.append([int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])])

        corrected_image, corrected_bboxes = rotate_image_and_bbox(image, bboxes, 修正する角度)

        cv2.imwrite(img_path, corrected_image)
        if os.path.exists(lbl_path) or len(corrected_bboxes) > 0:
            with open(lbl_path, 'w') as f:
                for bbox in corrected_bboxes:
                    f.write(f"{int(bbox[0])} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f} {bbox[4]:.6f}\n")

        processed_count += 1

    print(f"回転修正が完了しました。対象ファイル数: {processed_count}枚")
else:
    print("回転修正はスキップされました（角度が0、またはデータが存在しません）。")

回転修正処理を開始します (設定角度: 270度)...
回転修正が完了しました。対象ファイル数: 30枚


In [3]:
#@title 1.6. 画像とアノテーションのオーグメンテーション（オプション）
#@markdown 適用したいオーグメンテーションにチェックを入れてください。複数選択可能です。
#@markdown <br>※「前回のデータを削除する」にチェックを入れると、過去に作成した水増しデータを消去してから新しく作り直します。

前回のデータを削除する = False #@param {type:"boolean"}
Flip = True #@param {type:"boolean"}
Brightness = True #@param {type:"boolean"}
Exposure = False #@param {type:"boolean"}
Blur = False #@param {type:"boolean"}
Noise = False #@param {type:"boolean"}

import cv2
import numpy as np
import os
import glob

originals = '/content/original'
annotations = '/content/yolo/annotations'

# オーグメンテーションを識別する接尾辞
aug_suffixes = ['_fli', '_bri', '_exp', '_blu', '_noi']

if os.path.exists(originals):
    # 【追加機能】前回のオーグメンテーションデータを削除
    if 前回のデータを削除する:
        print("前回のオーグメンテーションデータを削除しています...")
        deleted_count = 0
        img_files = glob.glob(os.path.join(originals, "*"))
        for img_path in img_files:
            basename = os.path.basename(img_path)
            filename, ext = os.path.splitext(basename)
            # サフィックスが含まれるファイルを探して削除
            if any(filename.endswith(suf) for suf in aug_suffixes):
                os.remove(img_path)
                lbl_path = os.path.join(annotations, filename + ".txt")
                if os.path.exists(lbl_path):
                    os.remove(lbl_path)
                deleted_count += 1
        if deleted_count > 0:
            print(f"➜ {deleted_count} 枚の拡張データとアノテーションを削除しました。")
        else:
            print("➜ 削除する拡張データは見つかりませんでした。")

    any_checked = Flip or Brightness or Exposure or Blur or Noise

    if any_checked:
        print("オーグメンテーション処理を開始します...")
        # 削除後の最新のファイルリストを再取得
        img_files = glob.glob(os.path.join(originals, "*"))
        processed_count = 0

        for img_path in img_files:
            basename = os.path.basename(img_path)
            filename, ext = os.path.splitext(basename)

            if ext.lower() not in ['.jpg', '.jpeg', '.png', '.bmp']:
                continue

            # すでにオーグメンテーションされたファイルは元画像として扱わない（二重処理防止）
            if any(filename.endswith(suf) for suf in aug_suffixes):
                continue

            image = cv2.imread(img_path)
            if image is None:
                continue

            lbl_path = os.path.join(annotations, filename + ".txt")

            # ラベル(YOLO)読み込み
            bboxes = []
            if os.path.exists(lbl_path):
                with open(lbl_path, 'r') as f:
                    for line in f.readlines():
                        parts = line.strip().split()
                        if len(parts) == 5:
                            bboxes.append([int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])])

            # 保存処理の関数
            def save_augmented(aug_image, aug_bboxes, suffix):
                save_img_path = os.path.join(originals, f"{filename}_{suffix}{ext}")
                save_lbl_path = os.path.join(annotations, f"{filename}_{suffix}.txt")
                cv2.imwrite(save_img_path, aug_image)
                if os.path.exists(lbl_path) or len(aug_bboxes) > 0:
                    with open(save_lbl_path, 'w') as f:
                        for bbox in aug_bboxes:
                            f.write(f"{int(bbox[0])} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f} {bbox[4]:.6f}\n")
                return 1

            # 1. Flip (左右反転)
            if Flip:
                aug_image = cv2.flip(image, 1)
                aug_bboxes = [bbox.copy() for bbox in bboxes]
                for bbox in aug_bboxes:
                    bbox[1] = 1.0 - bbox[1]
                processed_count += save_augmented(aug_image, aug_bboxes, "fli")

            # 2. Brightness (輝度変更)
            if Brightness:
                aug_image = cv2.convertScaleAbs(image, alpha=1.0, beta=30)
                processed_count += save_augmented(aug_image, bboxes, "bri")

            # 3. Exposure (明度・コントラスト変更)
            if Exposure:
                aug_image = cv2.convertScaleAbs(image, alpha=1.3, beta=0)
                processed_count += save_augmented(aug_image, bboxes, "exp")

            # 4. Blur (ぼかし)
            if Blur:
                aug_image = cv2.GaussianBlur(image, (7, 7), 0)
                processed_count += save_augmented(aug_image, bboxes, "blu")

            # 5. Noise (ノイズ)
            if Noise:
                noise = np.random.randint(-15, 15, image.shape, dtype='int16')
                aug_image = np.clip(image.astype('int16') + noise, 0, 255).astype('uint8')
                processed_count += save_augmented(aug_image, bboxes, "noi")

        print(f"オーグメンテーションが完了しました。新たに {processed_count} 枚の拡張データを追加しました。")
    else:
        print("オーグメンテーションは選択されていないためスキップしました。")
else:
    print("データが存在しません。")

オーグメンテーション処理を開始します...
オーグメンテーションが完了しました。新たに 116 枚の拡張データを追加しました。


In [4]:
#@title 2. 画像の分割 (Train / Valid / Test)
#@markdown 全体から検証用(Valid)とテスト用(Test)に割り当てる割合をスライダーで設定してください。（残りが自動的に学習用になります）
#@markdown <br>※データリークを防ぐため、元画像単位で分割します。**Testにはオーグメンテーションデータは含まれません。**

validの割合 = 0.25 #@param {type:"slider", min:0, max:1, step:0.05}
testの割合 = 0.05 #@param {type:"slider", min:0, max:1, step:0.05}

import shutil
import os
from sklearn.model_selection import train_test_split
import math

originals = '/content/original'
annotations = '/content/yolo/annotations'
aug_suffixes = ['_fli', '_bri', '_exp', '_blu', '_noi']

dirs = [
    '/content/datasets/train/images', '/content/datasets/train/labels',
    '/content/datasets/valid/images', '/content/datasets/valid/labels',
    '/content/datasets/test/images', '/content/datasets/test/labels'
]

if os.path.exists('/content/datasets/'):
    shutil.rmtree('/content/datasets/')
for d in dirs:
    os.makedirs(d, exist_ok=True)

read_files_originals = sorted(os.listdir(originals))

# --- 元画像とオーグメンテーション画像をグループ化 ---
base_files = []
aug_map = {} # 元画像ファイル名(拡張子なし) -> 関連する拡張データのリスト

for f in read_files_originals:
    name, ext = os.path.splitext(f)
    is_aug = False

    # 拡張データかどうか判定
    for suf in aug_suffixes:
        if name.endswith(suf):
            is_aug = True
            base_name = name[:-len(suf)]
            if base_name not in aug_map:
                aug_map[base_name] = []
            aug_map[base_name].append(f)
            break

    # 元画像の場合
    if not is_aug:
        base_files.append(f)

total_bases = len(base_files)
if total_bases == 0:
    print("エラー: 元画像ファイルが見つかりません。")
else:
    valid_count = int(total_bases * validの割合)
    test_count = int(total_bases * testの割合)

    # まずベースとなる元画像を分割（データリーク防止）
    if valid_count > 0:
        base_rem, base_valid = train_test_split(
            base_files, test_size=valid_count, random_state=42
        )
    else:
        base_rem, base_valid = base_files, []

    if test_count > 0 and len(base_rem) > test_count:
        base_train, base_test = train_test_split(
            base_rem, test_size=test_count, random_state=42
        )
    else:
        base_train, base_test = base_rem, []

    # コピー用関数
    def copy_group(base_list, dest_img_dir, dest_lbl_dir, include_aug=True):
        copied_count = 0
        for base_f in base_list:
            base_name, ext = os.path.splitext(base_f)
            ann_f = base_name + ".txt"

            # 元画像のコピー
            shutil.copy(os.path.join(originals, base_f), dest_img_dir)
            if os.path.exists(os.path.join(annotations, ann_f)):
                shutil.copy(os.path.join(annotations, ann_f), dest_lbl_dir)
            copied_count += 1

            # 拡張データのコピー (Train, Validのみ)
            if include_aug and base_name in aug_map:
                for aug_f in aug_map[base_name]:
                    aug_ann = os.path.splitext(aug_f)[0] + ".txt"
                    shutil.copy(os.path.join(originals, aug_f), dest_img_dir)
                    if os.path.exists(os.path.join(annotations, aug_ann)):
                        shutil.copy(os.path.join(annotations, aug_ann), dest_lbl_dir)
                    copied_count += 1
        return copied_count

    # Testにはオーグメンテーションデータを含めない（include_aug=False）
    train_len = copy_group(base_train, '/content/datasets/train/images', '/content/datasets/train/labels', include_aug=True)
    valid_len = copy_group(base_valid, '/content/datasets/valid/images', '/content/datasets/valid/labels', include_aug=True)
    test_len = copy_group(base_test, '/content/datasets/test/images', '/content/datasets/test/labels', include_aug=False)

    gcd_val = math.gcd(train_len, valid_len) if train_len > 0 and valid_len > 0 else "計算不可"

    print("=== 分割結果 ===")
    print(f"学習用 (Train): {train_len}枚 (※水増しデータ含む)")
    print(f"検証用 (Valid): {valid_len}枚 (※水増しデータ含む)")
    print(f"テスト用 (Test): {test_len}枚 (※ピュアな元画像のみ)")
    print(f"最大公約数: {gcd_val} (※バッチサイズを決める際の目安になります)")

=== 分割結果 ===
学習用 (Train): 126枚 (※水増しデータ含む)
検証用 (Valid): 42枚 (※水増しデータ含む)
テスト用 (Test): 2枚 (※ピュアな元画像のみ)
最大公約数: 42 (※バッチサイズを決める際の目安になります)


#学習データに関する詳細ファイルを作成

In [5]:
#@title 3. 学習設定ファイル(data.yaml)の作成
#@markdown 検出したい項目（クラス名）を**カンマ区切り**で入力してください。数（nc）は自動計算されます。
#@markdown <br>例： `OK, NG` や `犬, 猫, 鳥`

クラス名 = "fruit, flower" #@param {type:"string"}

import yaml

names_list = [name.strip() for name in クラス名.split(",")]
nc = len(names_list)

yaml_data = {
    'train': '../train/images',
    'val': '../valid/images',
    'test': '../test/images',
    'nc': nc,
    'names': names_list
}

with open('/content/datasets/data.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False, allow_unicode=True)

print("以下の内容で data.yaml を作成しました：")
print("-" * 20)
print(f"クラス数 (nc): {nc}")
print(f"クラス名 (names): {names_list}")

以下の内容で data.yaml を作成しました：
--------------------
クラス数 (nc): 2
クラス名 (names): ['fruit', 'flower']


In [6]:
#@title 4. 完了したデータのダウンロード
#@markdown チェックを入れて実行すると、自動的にZIPファイルがダウンロードされます。
ダウンロードを実行する = True #@param {type:"boolean"}

import shutil
from google.colab import files

print("データセットをZIP化しています...")
shutil.make_archive('/content/datasets', 'zip', '/content/datasets')
print("ZIP化が完了しました。")

if ダウンロードを実行する:
    print("ダウンロードを開始します...")
    files.download("/content/datasets.zip")

データセットをZIP化しています...
ZIP化が完了しました。
ダウンロードを開始します...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>